#### Importing Libraries

In [1]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from collections import Counter

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score

# Defining reproducibility settings
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Configuring CPU execution
device = torch.device("cpu")

# Displaying environment information
print("PyTorch version:", torch.__version__)
print("Execution device:", device)
print("CPU threads:", torch.get_num_threads())

print("\nLibraries imported successfully.")

PyTorch version: 2.14.1+cpu
Execution device: cpu
CPU threads: 4

Libraries imported successfully.


#### Loading the Training Data

In [3]:
# Defining data paths
processed_dir = Path("../data/processed")
features_dir = processed_dir / "features"

# Loading previously saved datasets
train_df = pd.read_csv(processed_dir / "train.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

# Loading encoded target labels
y_train = np.load(features_dir / "y_train.npy")
y_test = np.load(features_dir / "y_test.npy")

# Extracting product titles
X_train_text = train_df["Product_Title"].astype(str).to_numpy()
X_test_text = test_df["Product_Title"].astype(str).to_numpy()

# Extracting group identifiers
train_groups = train_df["Split_Group"].to_numpy()
test_groups = test_df["Split_Group"].to_numpy()

# Validating dataset alignment
assert len(X_train_text) == len(y_train)
assert len(X_test_text) == len(y_test)
assert set(train_groups).isdisjoint(set(test_groups))

# Displaying dataset summary
print("TextCNN Dataset Summary:")
print(f"Training samples: {len(X_train_text):,}")
print(f"Test samples: {len(X_test_text):,}")
print(f"Number of classes: {len(np.unique(y_train))}")
print(f"Training groups: {len(np.unique(train_groups)):,}")
print(f"Test groups: {len(np.unique(test_groups)):,}")

print("\nTraining Data Preview:")
display(train_df.head())

print("\nTraining data loaded and validated successfully.")

TextCNN Dataset Summary:
Training samples: 28,919
Test samples: 6,391
Number of classes: 10
Training groups: 9,577
Test groups: 2,395

Training Data Preview:


,Product_ID,Product_Title,Merchant_ID,Cluster_ID,Cluster_Label,Category_ID,Category_Label,Split_Group
0,1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones,1
1,2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones,1
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones,1
3,4,apple iphone 8 plus 64gb space grey,4,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones,1
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones,1



Training data loaded and validated successfully.


#### Building the Text Vocabulary

In [4]:
# Defining vocabulary parameters
MAX_VOCAB_SIZE = 15000
MAX_SEQUENCE_LENGTH = 40

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

# Defining a lightweight tokenizer
def tokenize_text(text):
    return str(text).lower().split()

# Counting tokens in the training data
token_counts = Counter()

for title in X_train_text:
    token_counts.update(tokenize_text(title))

# Initializing vocabulary with special tokens
vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

# Adding the most frequent tokens
for token, count in token_counts.most_common(MAX_VOCAB_SIZE - 2):
    vocab[token] = len(vocab)

# Displaying vocabulary summary
print("TextCNN Vocabulary Summary:")
print(f"Unique tokens: {len(token_counts):,}")
print(f"Vocabulary size: {len(vocab):,}")
print(f"Maximum sequence length: {MAX_SEQUENCE_LENGTH}")

print("\nTop 15 Most Frequent Tokens:")
display(
    pd.DataFrame(
        token_counts.most_common(15),
        columns=["Token", "Frequency"]
    )
)

print("\nVocabulary created successfully.")

TextCNN Vocabulary Summary:
Unique tokens: 18,145
Vocabulary size: 15,000
Maximum sequence length: 40

Top 15 Most Frequent Tokens:


,Token,Frequency
0,fridge,5210
1,freezer,4947
2,white,4389
3,in,3607
4,black,3260
5,free,2967
6,a,2915
7,bosch,2604
8,tv,2476
9,integrated,2236



Vocabulary created successfully.


**Observations on Text Vocabulary:**

- The training dataset contains **18,145 unique tokens**, with the vocabulary restricted to the **15,000 most frequent tokens** for computational efficiency.
- Frequently occurring terms such as **fridge, freezer, Bosch, Intel, and processor** reflect product categories, brands, and technical specifications.
- A maximum sequence length of **40 tokens** has been selected to standardize model inputs while keeping the TextCNN architecture lightweight.

#### Encoding and Padding Product Titles

In [5]:
# Defining text encoding function
def encode_text(text, vocabulary, max_length=MAX_SEQUENCE_LENGTH):

    tokens = tokenize_text(text)

    # Converting tokens into integer IDs
    token_ids = [
        vocabulary.get(token, vocabulary[UNK_TOKEN])
        for token in tokens[:max_length]
    ]

    # Padding shorter sequences
    token_ids += [vocabulary[PAD_TOKEN]] * (
        max_length - len(token_ids)
    )

    return token_ids


# Encoding training product titles
X_train_seq = np.array(
    [
        encode_text(title, vocab)
        for title in X_train_text
    ],
    dtype=np.int64
)

# Encoding test product titles
X_test_seq = np.array(
    [
        encode_text(title, vocab)
        for title in X_test_text
    ],
    dtype=np.int64
)

# Validating encoded sequences
assert X_train_seq.shape == (
    len(X_train_text),
    MAX_SEQUENCE_LENGTH
)

assert X_test_seq.shape == (
    len(X_test_text),
    MAX_SEQUENCE_LENGTH
)

# Displaying encoding summary
print("TextCNN Sequence Encoding Summary:")
print(f"Training sequences: {X_train_seq.shape}")
print(f"Test sequences: {X_test_seq.shape}")
print(f"Sequence data type: {X_train_seq.dtype}")

print("\nOriginal Product Title:")
print(X_train_text[0])

print("\nEncoded Product Title:")
print(X_train_seq[0])

print("\nEncoding and padding completed successfully.")

TextCNN Sequence Encoding Summary:
Training sequences: (28919, 40)
Test sequences: (6391, 40)
Sequence data type: int64

Original Product Title:
apple iphone 8 plus 64gb silver

Encoded Product Title:
[149 123  57 148 142  34   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0]

Encoding and padding completed successfully.


**Observations on Sequence Encoding:**

- All **28,919 training titles** and **6,391 test titles** were successfully encoded into fixed-length sequences of **40 tokens**.
- The sample product title was converted into integer token IDs, with **zero-padding** applied to maintain a consistent input length.
- The resulting integer sequences are suitable for PyTorch's embedding layer, enabling TextCNN to learn representations directly from product titles.

#### Creating the PyTorch Dataset

In [6]:
# Defining a custom PyTorch dataset
class ProductTitleDataset(Dataset):

    def __init__(self, sequences, labels):
        self.sequences = torch.as_tensor(
            sequences,
            dtype=torch.long
        )

        self.labels = torch.as_tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


# Creating a training dataset for validation
train_dataset = ProductTitleDataset(
    X_train_seq,
    y_train
)

# Inspecting the dataset
sample_sequence, sample_label = train_dataset[0]

print("PyTorch Dataset Summary:")
print(f"Training samples: {len(train_dataset):,}")
print(f"Sequence shape: {sample_sequence.shape}")
print(f"Sequence dtype: {sample_sequence.dtype}")
print(f"Label dtype: {sample_label.dtype}")

print("\nSample Label:")
print(sample_label.item())

print("\nPyTorch dataset created successfully.")

PyTorch Dataset Summary:
Training samples: 28,919
Sequence shape: torch.Size([40])
Sequence dtype: torch.int64
Label dtype: torch.int64

Sample Label:
7

PyTorch dataset created successfully.


**Observations on PyTorch Dataset:**

- The custom PyTorch dataset successfully contains **28,919 training samples**, each represented by a sequence of **40 tokens**.
- Product titles and category labels are stored as **64-bit integer tensors**, ensuring compatibility with PyTorch embedding layers and classification loss functions.
- The dataset structure supports efficient batch processing and can be reused within group-aware cross-validation.

#### Defining the TextCNN Architecture

In [7]:
import torch.nn.functional as F

# Defining TextCNN architecture
class TextCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        num_filters=64,
        kernel_sizes=(3, 4, 5),
        num_classes=10,
        dropout=0.3
    ):
        super().__init__()

        # Learning token embeddings
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # Extracting local text patterns
        self.convolutions = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=kernel_size
            )
            for kernel_size in kernel_sizes
        ])

        # Applying dropout regularization
        self.dropout = nn.Dropout(dropout)

        # Predicting product categories
        self.classifier = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

    def forward(self, x):

        # Shape: (batch_size, sequence_length, embedding_dim)
        embedded = self.embedding(x)

        # Shape: (batch_size, embedding_dim, sequence_length)
        embedded = embedded.permute(0, 2, 1)

        # Applying convolution and global max pooling
        pooled_outputs = []

        for convolution in self.convolutions:
            features = F.relu(convolution(embedded))
            pooled = F.max_pool1d(
                features,
                kernel_size=features.shape[2]
            ).squeeze(2)

            pooled_outputs.append(pooled)

        # Combining convolutional features
        combined = torch.cat(pooled_outputs, dim=1)

        # Applying dropout and classification
        output = self.classifier(
            self.dropout(combined)
        )

        return output


# Initializing TextCNN
textcnn_model = TextCNN(
    vocab_size=len(vocab),
    num_classes=len(np.unique(y_train))
).to(device)

# Calculating trainable parameters
trainable_parameters = sum(
    parameter.numel()
    for parameter in textcnn_model.parameters()
    if parameter.requires_grad
)

# Displaying model architecture
print("TextCNN Architecture:")
print(textcnn_model)

print(f"\nTrainable parameters: {trainable_parameters:,}")

# Validating output dimensions
textcnn_model.eval()

with torch.no_grad():
    sample_input = torch.as_tensor(
        X_train_seq[:4],
        dtype=torch.long,
        device=device
    )

    sample_output = textcnn_model(sample_input)

print(f"Sample input shape: {sample_input.shape}")
print(f"Sample output shape: {sample_output.shape}")

assert sample_output.shape == (
    4,
    len(np.unique(y_train))
)

print("\nTextCNN architecture validated successfully.")

TextCNN Architecture:
TextCNN(
  (embedding): Embedding(15000, 64, padding_idx=0)
  (convolutions): ModuleList(
    (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,))
    (1): Conv1d(64, 64, kernel_size=(4,), stride=(1,))
    (2): Conv1d(64, 64, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=192, out_features=10, bias=True)
)

Trainable parameters: 1,011,274
Sample input shape: torch.Size([4, 40])
Sample output shape: torch.Size([4, 10])

TextCNN architecture validated successfully.


**Observations on TextCNN Architecture:**

- TextCNN contains **1,011,274 trainable parameters**, using 64-dimensional embeddings and three parallel convolutional layers.
- Convolutional filters with kernel sizes **3, 4, and 5** capture local word patterns, while global max pooling combines the extracted features.
- The model successfully transforms input sequences of **40 tokens** into prediction scores for **10 product categories**, confirming architectural compatibility.

#### Configuring the TextCNN Training Process

In [9]:
from copy import deepcopy

# Defining training configuration
BATCH_SIZE = 128
LEARNING_RATE = 0.001
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3


# Defining the training function
def train_textcnn(
    model,
    train_loader,
    valid_loader,
    learning_rate=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE
):

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    best_f1 = -1.0
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0

    training_history = []

    for epoch in range(1, max_epochs + 1):

        # Training phase
        model.train()
        running_loss = 0.0

        for sequences, labels in train_loader:

            sequences = sequences.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(sequences)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += (
                loss.item() * sequences.size(0)
            )

        train_loss = running_loss / len(train_loader.dataset)

        # Validation phase
        model.eval()

        valid_predictions = []
        valid_labels = []

        with torch.no_grad():

            for sequences, labels in valid_loader:

                sequences = sequences.to(device)

                outputs = model(sequences)

                predictions = torch.argmax(
                    outputs,
                    dim=1
                )

                valid_predictions.extend(
                    predictions.cpu().numpy()
                )

                valid_labels.extend(
                    labels.numpy()
                )

        valid_f1 = f1_score(
            valid_labels,
            valid_predictions,
            average="macro",
            zero_division=0
        )

        training_history.append({
            "Epoch": epoch,
            "Train_Loss": train_loss,
            "Validation_Macro_F1": valid_f1
        })

        print(
            f"Epoch {epoch:02d}/{max_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Validation Macro F1: {valid_f1:.4f}"
        )

        # Tracking best model
        if valid_f1 > best_f1 + 1e-4:

            best_f1 = valid_f1
            best_epoch = epoch

            best_state = deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        # Applying early stopping
        if epochs_without_improvement >= patience:

            print(
                f"Early stopping triggered at epoch {epoch}."
            )

            break

    # Restoring best model weights
    model.load_state_dict(best_state)

    print(
        f"\nBest Epoch: {best_epoch} | "
        f"Best Validation Macro F1: {best_f1:.4f}"
    )

    return model, pd.DataFrame(training_history)

#### Configuring Group-Aware Cross-Validation

In [10]:
from sklearn.model_selection import StratifiedGroupKFold

# Configuring outer cross-validation
textcnn_outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Configuring inner validation splitting
textcnn_inner_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Preparing outer cross-validation splits
textcnn_outer_splits = list(
    textcnn_outer_cv.split(
        X_train_text,
        y_train,
        groups=train_groups
    )
)

# Validating group separation across folds
fold_summary = []

for fold, (outer_train_idx, outer_valid_idx) in enumerate(
    textcnn_outer_splits,
    start=1
):

    outer_train_groups = set(train_groups[outer_train_idx])
    outer_valid_groups = set(train_groups[outer_valid_idx])

    # Selecting an inner validation split
    inner_train_rel, inner_valid_rel = next(
        textcnn_inner_cv.split(
            X_train_text[outer_train_idx],
            y_train[outer_train_idx],
            groups=train_groups[outer_train_idx]
        )
    )

    inner_train_idx = outer_train_idx[inner_train_rel]
    inner_valid_idx = outer_train_idx[inner_valid_rel]

    # Checking group separation
    assert set(train_groups[inner_train_idx]).isdisjoint(
        set(train_groups[inner_valid_idx])
    )

    assert outer_train_groups.isdisjoint(
        outer_valid_groups
    )

    assert set(train_groups[inner_train_idx]).isdisjoint(
        outer_valid_groups
    )

    assert set(train_groups[inner_valid_idx]).isdisjoint(
        outer_valid_groups
    )

    fold_summary.append({
        "Fold": fold,
        "Inner_Train_Samples": len(inner_train_idx),
        "Early_Stopping_Samples": len(inner_valid_idx),
        "Outer_Validation_Samples": len(outer_valid_idx),
        "Group_Overlap": 0
    })

# Displaying cross-validation summary
textcnn_fold_summary_df = pd.DataFrame(fold_summary)

print("TextCNN Cross-Validation Summary:")
display(textcnn_fold_summary_df)

print("\nGroup-aware cross-validation configured successfully.")

TextCNN Cross-Validation Summary:


,Fold,Inner_Train_Samples,Early_Stopping_Samples,Outer_Validation_Samples,Group_Overlap
0,1,18231,4953,5735,0
1,2,18228,4954,5737,0
2,3,18230,4953,5736,0
3,4,18354,4589,5976,0
4,5,18230,4954,5735,0



Group-aware cross-validation configured successfully.


**Observations on Group-Aware Cross-Validation:**

- Five-fold StratifiedGroupKFold validation was configured, with approximately **5,700–6,000 samples** reserved for evaluation in each outer fold.
- Separate early-stopping validation partitions contain approximately **4,600–5,000 samples**, ensuring that outer-validation labels are not used during training.
- **Zero group overlap** was confirmed across all folds, reducing the risk of data leakage between related product listings.

#### Preparing Fold-Specific TextCNN Datasets

In [11]:
# Defining fold-specific dataset preparation
def prepare_textcnn_fold(
    inner_train_idx,
    inner_valid_idx,
    outer_valid_idx
):

    # Extracting fold-specific product titles
    inner_train_text = X_train_text[inner_train_idx]
    inner_valid_text = X_train_text[inner_valid_idx]
    outer_valid_text = X_train_text[outer_valid_idx]

    # Building vocabulary from inner-training data only
    fold_token_counts = Counter()

    for title in inner_train_text:
        fold_token_counts.update(tokenize_text(title))

    fold_vocab = {
        PAD_TOKEN: 0,
        UNK_TOKEN: 1
    }

    for token, count in fold_token_counts.most_common(
        MAX_VOCAB_SIZE - 2
    ):
        fold_vocab[token] = len(fold_vocab)

    # Encoding product titles using fold-specific vocabulary
    def encode_titles(titles):
        return np.array(
            [
                encode_text(title, fold_vocab)
                for title in titles
            ],
            dtype=np.int64
        )

    X_inner_train = encode_titles(inner_train_text)
    X_inner_valid = encode_titles(inner_valid_text)
    X_outer_valid = encode_titles(outer_valid_text)

    # Creating PyTorch datasets
    inner_train_dataset = ProductTitleDataset(
        X_inner_train,
        y_train[inner_train_idx]
    )

    inner_valid_dataset = ProductTitleDataset(
        X_inner_valid,
        y_train[inner_valid_idx]
    )

    outer_valid_dataset = ProductTitleDataset(
        X_outer_valid,
        y_train[outer_valid_idx]
    )

    # Creating data loaders
    train_loader = DataLoader(
        inner_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    valid_loader = DataLoader(
        inner_valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    outer_valid_loader = DataLoader(
        outer_valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    return (
        fold_vocab,
        train_loader,
        valid_loader,
        outer_valid_loader
    )


# Testing dataset preparation on the first fold
outer_train_idx, outer_valid_idx = textcnn_outer_splits[0]

inner_train_rel, inner_valid_rel = next(
    textcnn_inner_cv.split(
        X_train_text[outer_train_idx],
        y_train[outer_train_idx],
        groups=train_groups[outer_train_idx]
    )
)

inner_train_idx = outer_train_idx[inner_train_rel]
inner_valid_idx = outer_train_idx[inner_valid_rel]

(
    sample_fold_vocab,
    sample_train_loader,
    sample_valid_loader,
    sample_outer_loader
) = prepare_textcnn_fold(
    inner_train_idx,
    inner_valid_idx,
    outer_valid_idx
)

# Displaying dataset preparation summary
print("Fold 1 Dataset Preparation:")
print(f"Vocabulary size: {len(sample_fold_vocab):,}")
print(f"Training samples: {len(sample_train_loader.dataset):,}")
print(f"Early-stopping samples: {len(sample_valid_loader.dataset):,}")
print(f"Outer-validation samples: {len(sample_outer_loader.dataset):,}")

# Checking batch dimensions
sample_sequences, sample_labels = next(
    iter(sample_train_loader)
)

print(f"\nBatch sequences: {sample_sequences.shape}")
print(f"Batch labels: {sample_labels.shape}")

print("\nFold-specific TextCNN datasets prepared successfully.")

Fold 1 Dataset Preparation:
Vocabulary size: 13,080
Training samples: 18,231
Early-stopping samples: 4,953
Outer-validation samples: 5,735

Batch sequences: torch.Size([128, 40])
Batch labels: torch.Size([128])

Fold-specific TextCNN datasets prepared successfully.


**Observations on Fold-Specific Dataset Preparation:**

- Fold 1 contains **18,231 training samples**, **4,953 early-stopping samples**, and **5,735 outer-validation samples**.
- The fold-specific vocabulary contains **13,080 tokens**, constructed exclusively from the inner-training partition to prevent vocabulary leakage.
- Data loaders successfully produce batches of **128 samples with 40 tokens each**, confirming compatibility with the TextCNN architecture.

#### Training and Evaluating TextCNN Using Group-Aware Cross-Validation

In [13]:
# Storing fold-level results and training histories
textcnn_fold_results = []
textcnn_training_histories = []

# Training and evaluating TextCNN across five folds
for fold, (outer_train_idx, outer_valid_idx) in enumerate(
    textcnn_outer_splits,
    start=1
):

    print(f"\n{'=' * 55}")
    print(f"TextCNN - Fold {fold}/5")
    print(f"{'=' * 55}")

    # Creating inner training and early-stopping partitions
    inner_train_rel, inner_valid_rel = next(
        textcnn_inner_cv.split(
            X_train_text[outer_train_idx],
            y_train[outer_train_idx],
            groups=train_groups[outer_train_idx]
        )
    )

    inner_train_idx = outer_train_idx[inner_train_rel]
    inner_valid_idx = outer_train_idx[inner_valid_rel]

    # Preparing fold-specific vocabulary and data loaders
    (
        fold_vocab,
        train_loader,
        valid_loader,
        outer_valid_loader
    ) = prepare_textcnn_fold(
        inner_train_idx,
        inner_valid_idx,
        outer_valid_idx
    )

    # Initializing a fresh model for each fold
    torch.manual_seed(RANDOM_STATE + fold)

    fold_model = TextCNN(
        vocab_size=len(fold_vocab),
        num_classes=len(np.unique(y_train))
    ).to(device)

    # Training with early stopping
    fold_model, fold_history = train_textcnn(
        model=fold_model,
        train_loader=train_loader,
        valid_loader=valid_loader
    )

    fold_history["Fold"] = fold
    textcnn_training_histories.append(fold_history)

    # Evaluating on the outer-validation partition
    fold_model.eval()

    outer_predictions = []
    outer_labels = []

    with torch.no_grad():

        for sequences, labels in outer_valid_loader:

            sequences = sequences.to(device)

            outputs = fold_model(sequences)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            outer_predictions.extend(
                predictions.cpu().numpy()
            )

            outer_labels.extend(
                labels.numpy()
            )

    # Calculating fold-level performance
    fold_accuracy = accuracy_score(
        outer_labels,
        outer_predictions
    )

    fold_macro_f1 = f1_score(
        outer_labels,
        outer_predictions,
        average="macro",
        zero_division=0
    )

    fold_weighted_f1 = f1_score(
        outer_labels,
        outer_predictions,
        average="weighted",
        zero_division=0
    )

    textcnn_fold_results.append({
        "Fold": fold,
        "Accuracy": fold_accuracy,
        "Macro_F1": fold_macro_f1,
        "Weighted_F1": fold_weighted_f1,
        "Best_Epoch": int(
            fold_history.loc[
                fold_history["Validation_Macro_F1"].idxmax(),
                "Epoch"
            ]
        )
    })

    print(
        f"\nFold {fold} Results | "
        f"Accuracy: {fold_accuracy:.4f} | "
        f"Macro F1: {fold_macro_f1:.4f} | "
        f"Weighted F1: {fold_weighted_f1:.4f}"
    )

    # Releasing fold-specific resources
    del fold_model
    del train_loader, valid_loader, outer_valid_loader

# Creating fold-level results DataFrame
textcnn_cv_results_df = pd.DataFrame(
    textcnn_fold_results
)

# Summarizing cross-validation performance
textcnn_results = {
    "Model": "TextCNN",
    "Feature_Representation": "Learned Word Embeddings",
    "Accuracy": textcnn_cv_results_df["Accuracy"].mean(),
    "Macro_F1": textcnn_cv_results_df["Macro_F1"].mean(),
    "Weighted_F1": textcnn_cv_results_df["Weighted_F1"].mean()
}

textcnn_results_df = pd.DataFrame([textcnn_results])

# Displaying individual model results
print("\nTextCNN Cross-Validation Results:")
display(textcnn_results_df.round(4))

# Displaying fold-level performance
print("\nTextCNN Fold-Level Results:")
display(textcnn_cv_results_df.round(4))

print("\nTextCNN cross-validation completed successfully.")


TextCNN - Fold 1/5
Epoch 01/15 | Train Loss: 0.8283 | Validation Macro F1: 0.8930
Epoch 02/15 | Train Loss: 0.2941 | Validation Macro F1: 0.9134
Epoch 03/15 | Train Loss: 0.2107 | Validation Macro F1: 0.9185
Epoch 04/15 | Train Loss: 0.1639 | Validation Macro F1: 0.9231
Epoch 05/15 | Train Loss: 0.1337 | Validation Macro F1: 0.9289
Epoch 06/15 | Train Loss: 0.1112 | Validation Macro F1: 0.9317
Epoch 07/15 | Train Loss: 0.0929 | Validation Macro F1: 0.9330
Epoch 08/15 | Train Loss: 0.0784 | Validation Macro F1: 0.9249
Epoch 09/15 | Train Loss: 0.0643 | Validation Macro F1: 0.9343
Epoch 10/15 | Train Loss: 0.0561 | Validation Macro F1: 0.9281
Epoch 11/15 | Train Loss: 0.0479 | Validation Macro F1: 0.9341
Epoch 12/15 | Train Loss: 0.0402 | Validation Macro F1: 0.9342
Early stopping triggered at epoch 12.

Best Epoch: 9 | Best Validation Macro F1: 0.9343

Fold 1 Results | Accuracy: 0.9334 | Macro F1: 0.9331 | Weighted F1: 0.9331

TextCNN - Fold 2/5
Epoch 01/15 | Train Loss: 0.8257 | Valid

,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,TextCNN,Learned Word Embeddings,0.932,0.9333,0.9332



TextCNN Fold-Level Results:


,Fold,Accuracy,Macro_F1,Weighted_F1,Best_Epoch
0,1,0.9334,0.9331,0.9331,9
1,2,0.9264,0.9291,0.9272,6
2,3,0.9355,0.9365,0.9367,10
3,4,0.9344,0.9328,0.9349,7
4,5,0.9303,0.9348,0.9339,12



TextCNN cross-validation completed successfully.


**Observations on TextCNN (Learned Word Embeddings):**

- TextCNN achieved **93.20% accuracy** and a **Macro F1-score of 0.9333** across five group-aware cross-validation folds.
- Fold-level Macro F1 ranged from **0.9291 to 0.9365**, indicating relatively consistent predictive performance across validation partitions.
- TextCNN underperformed leading character-level TF-IDF classifiers, with Ridge Classifier achieving **0.9877 Macro F1**, suggesting that character-level representations are more effective for this product-title classification task.

#### Logging TextCNN Cross-Validation Results in MLflow

In [14]:
import mlflow

# Configuring MLflow tracking
tracking_db = Path("../mlflow.db").resolve()

mlflow.set_tracking_uri(
    f"sqlite:///{tracking_db.as_posix()}"
)

mlflow.set_experiment("Product Taxonomy Intelligence")

# Combining training histories
textcnn_history_df = pd.concat(
    textcnn_training_histories,
    ignore_index=True
)

# Logging TextCNN experiment
with mlflow.start_run(run_name="TextCNN_Word_Embeddings"):

    # Logging model configuration
    mlflow.log_params({
        "model": "TextCNN",
        "feature_representation": "Learned Word Embeddings",
        "framework": "PyTorch",
        "vocabulary_limit": MAX_VOCAB_SIZE,
        "sequence_length": MAX_SEQUENCE_LENGTH,
        "embedding_dim": 64,
        "num_filters": 64,
        "kernel_sizes": "3,4,5",
        "dropout": 0.3,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "outer_cv_strategy": "StratifiedGroupKFold",
        "outer_cv_folds": 5,
        "inner_cv_strategy": "StratifiedGroupKFold",
        "inner_cv_folds": 5,
        "execution_device": str(device),
        "random_state": RANDOM_STATE
    })

    # Logging cross-validation metrics
    for metric in ["Accuracy", "Macro_F1", "Weighted_F1"]:

        values = textcnn_cv_results_df[metric]

        mlflow.log_metric(
            f"cv_{metric.lower()}_mean",
            values.mean()
        )

        mlflow.log_metric(
            f"cv_{metric.lower()}_std",
            values.std()
        )

    # Logging individual model results
    mlflow.log_table(
        textcnn_results_df,
        artifact_file="textcnn_results.json"
    )

    # Logging fold-level results
    mlflow.log_table(
        textcnn_cv_results_df,
        artifact_file="textcnn_fold_results.json"
    )

    # Logging training histories
    mlflow.log_table(
        textcnn_history_df,
        artifact_file="textcnn_training_history.json"
    )

print("TextCNN experiment logged successfully in MLflow.")

TextCNN experiment logged successfully in MLflow.
